
# LangChain Code Building Blocks

## Overview

In this notebook, we will explore the fundamental building blocks of LangChain through practical Python examples.

The main concepts covered are:

1. Installing required packages
2. Configuring an OpenAI API key
3. Initializing an LLM using LangChain
4. Invoking a model directly
5. Creating reusable Prompt Templates
6. Creating Chat Prompt Templates
7. Building chains using LangChain Expression Language (LCEL)
8. Using the `invoke()` method
9. Understanding memory and conversation context

The goal is to understand how LangChain provides a unified interface for working with language models and how its components can be composed into reusable pipelines.

---

## 1. Prerequisites

Before running this notebook, you need:

- Python installed
- A LangChain environment
- An OpenAI API key
- Basic Python knowledge

Install the required packages using:

```bash
pip install -U langchain langchain-openai python-dotenv
````

Depending on the LangChain version and model provider you use, additional packages may be required.

---

## 2. Configure the OpenAI API Key

The examples in this notebook use an OpenAI model.

Never hard-code your API key directly into your Python source code.

Instead, create a `.env` file in the project directory:

```text
OPENAI_API_KEY=your_api_key_here
```

Then load the environment variable using `python-dotenv`.

Example:

```python
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OpenAI API Key Found")
else:
    print("OpenAI API Key Not Found")
```

### Important Security Note

API keys are sensitive credentials.

Do not:

* Commit `.env` files to GitHub
* Share API keys publicly
* Put API keys directly inside notebooks that will be published
* Include API keys in screenshots

Add `.env` to your `.gitignore` file:

```text
.env
```

---

## 3. Initialize a Chat Model

The model is the reasoning engine of a LangChain application.

LangChain provides a standardized interface for interacting with different model providers.

For example, the same general LangChain approach can be used with models from providers such as:

* OpenAI
* Anthropic
* Google
* Other supported providers

The exact model name depends on the provider and the models available to your account.

Example using an OpenAI chat model:

```python
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5.5",
    temperature=0
)
```

> Note: Model availability changes over time. If `gpt-5.5` is not available in your environment, replace it with a model available to your OpenAI account.

---

## 4. Invoking the Model Directly

The simplest way to use a LangChain model is to call its `invoke()` method.

```python
response = model.invoke("What is LangChain?")

print(response.content)
```

The model returns an AI message object.

The `content` property contains the actual text generated by the model.

Conceptually:

```text
User Question
      |
      v
LangChain Model
      |
      v
AIMessage
      |
      v
response.content
      |
      v
Plain Text Answer
```

The important idea is that `invoke()` is the standard way to execute a LangChain runnable component.

---

## 5. Prompt Templates

A Prompt Template is a reusable template for generating prompts dynamically.

Instead of writing a new prompt every time, we define a template with placeholders.

For example:

```python
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "Explain {topic} to a beginner."
)
```

Now we can provide a value for `topic`:

```python
formatted_prompt = prompt.invoke({
    "topic": "AI agents"
})

print(formatted_prompt)
```

We can reuse the same template with different topics:

```python
prompt.invoke({"topic": "LangChain"})
prompt.invoke({"topic": "Python"})
prompt.invoke({"topic": "Machine Learning"})
```

This makes prompts reusable and dynamic.

### Mental Model

Think of a Prompt Template like a Python function.

A Python function:

```python
def greet(name):
    return f"Hello {name}"
```

A Prompt Template:

```python
prompt = PromptTemplate.from_template(
    "Explain {topic} to a beginner."
)
```

Both use inputs to produce dynamically generated output.

---

## 6. Chat Prompt Templates

Chat models typically work with different message roles.

The most common roles are:

### System Message

Defines the behavior, instructions, personality, or constraints of the model.

Example:

```text
You are a helpful programming instructor.
```

### Human Message

Represents the user's input.

Example:

```text
Explain Python loops.
```

### AI Message

Represents a previous response generated by the AI.

Chat Prompt Templates allow us to structure these messages.

Example:

```python
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful programming instructor."
    ),
    (
        "human",
        "Explain {concept} with a simple Python example."
    )
])
```

Now we can provide a value:

```python
messages = chat_prompt.invoke({
    "concept": "for loops"
})

print(messages)
```

This produces structured chat messages that can be passed to the model.

---

## 7. Building Chains

One of LangChain's most important ideas is the concept of a chain.

A chain connects multiple processing steps together.

A simple chain can look like:

```text
Prompt Template
       |
       v
     Model
       |
       v
Output Parser
       |
       v
Final String
```

For example:

```python
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful Python instructor."
    ),
    (
        "human",
        "Explain {concept} with a simple Python example."
    )
])
```

We can connect the prompt to the model and then to an output parser.

```python
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

chain = prompt | model | parser
```

The pipe operator `|` connects the components.

This syntax is part of LangChain Expression Language (LCEL).

The chain means:

```text
Input
  |
  v
Prompt Template
  |
  v
LLM
  |
  v
String Output Parser
  |
  v
Final Answer
```

---

## 8. Invoking a Chain

Once the chain is created, we can invoke it.

```python
result = chain.invoke({
    "concept": "for loops"
})

print(result)
```

The execution process is approximately:

1. The input `for loops` is provided.
2. The Prompt Template fills the `{concept}` placeholder.
3. The resulting prompt is sent to the model.
4. The model generates a response.
5. `StrOutputParser` extracts the plain text.
6. The final result is returned.

Conceptually:

```text
{"concept": "for loops"}
          |
          v
Explain for loops with a simple Python example.
          |
          v
       LLM Model
          |
          v
      AI Response
          |
          v
   StringOutputParser
          |
          v
      Plain String
```

---

## 9. Why Chains Are Powerful

Chains allow us to compose reusable components.

Instead of writing all application logic manually, we can build a pipeline:

```python
chain = prompt | model | parser
```

Each component has a specific responsibility:

* **Prompt**: Structures and prepares the input
* **Model**: Generates the response
* **Parser**: Converts the response into the required format

This modular architecture makes LangChain applications easier to build and maintain.

---

# 10. Memory and Conversation Context

Large Language Models are generally stateless between separate API calls.

For example, imagine these two calls:

```text
User: My name is Alex.
```

Then later:

```text
User: What is my name?
```

If the previous message is not provided as context, the model may not know that the user's name is Alex.

Memory solves this problem by storing previous messages and including them in future model requests.

---

## 11. Creating an In-Memory Store

For a simple demonstration, we can store messages in Python memory.

```python
from langchain_core.messages import HumanMessage, AIMessage

memory = [
    HumanMessage(
        content="My name is Alex. I am learning LangChain."
    ),
    AIMessage(
        content="Nice to meet you, Alex. LangChain is a great choice for learning LLM application development."
    )
]
```

The memory contains both:

* Human messages
* AI messages

---

## 12. Injecting Memory into a Prompt

We can create a chat prompt that accepts previous messages.

```python
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

memory_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant."
    ),
    MessagesPlaceholder(
        variable_name="history"
    ),
    (
        "human",
        "{question}"
    )
])
```

The `MessagesPlaceholder` allows us to dynamically insert previous conversation messages.

---

## 13. Creating a Memory Chain

We can connect the memory-aware prompt to the model and output parser.

```python
memory_chain = memory_prompt | model | parser
```

Then invoke the chain:

```python
answer = memory_chain.invoke({
    "history": memory,
    "question": "What was my name again?"
})

print(answer)
```

The model receives the previous conversation as context and can answer:

```text
Your name is Alex.
```

The important point is that the model did not magically remember the information.

The application explicitly provided the previous conversation as context.

---

## 14. Memory Mental Model

The process looks like this:

```text
Previous Conversation
        |
        v
    Memory Store
        |
        v
MessagesPlaceholder
        |
        v
    Chat Prompt
        |
        v
       LLM
        |
        v
   Final Answer
```

Memory is therefore an application-level mechanism for maintaining context.

---

# 15. Key Takeaways

In this practical walkthrough, we learned:

### 1. Models

The model is the reasoning engine of a LangChain application.

```python
model = ChatOpenAI(...)
```

### 2. `invoke()`

The `invoke()` method executes a LangChain component.

```python
response = model.invoke("What is LangChain?")
```

### 3. Prompt Templates

Prompt templates create reusable prompts with dynamic variables.

```python
PromptTemplate.from_template(
    "Explain {topic} to a beginner."
)
```

### 4. Chat Prompt Templates

Chat prompts organize messages using roles such as:

* System
* Human
* AI

### 5. Chains

Chains connect multiple components using the pipe operator.

```python
chain = prompt | model | parser
```

### 6. Output Parsers

Output parsers convert model responses into convenient formats.

```python
StrOutputParser()
```

### 7. Memory

Memory provides previous conversation context to otherwise stateless model calls.

---

# Final Concept

The core LangChain building blocks can be visualized as:

```text
                    LangChain Application
                           |
        +------------------+------------------+
        |                  |                  |
        v                  v                  v
      Model             Prompt             Chain
   (Reasoning)        (Instructions)     (Pipeline)
        |                  |                  |
        +------------------+------------------+
                           |
                           v
                         Memory
                    (Conversation Context)
```

The key idea is that LangChain allows developers to combine these building blocks into larger applications.

Instead of simply calling an LLM, developers can create structured, reusable pipelines that connect prompts, models, tools, memory, and application logic.

````